In [1]:
import sys
sys.path.append('../code')

import pandas as pd
import numpy as np
import pysam

from tqdm.notebook import tqdm
from preprocessing_tools import *

tqdm.pandas()

In [2]:
contexts = pd.read_csv('../data/COSMIC/COSMIC_v3.4_SBS_GRCh38.txt', usecols=['Type'], sep='\t')
assembly = pd.read_json('../data/ncbi_genome/ncbi_dataset/data/sequence_report.jsonl', lines=True)
chr_id = dict(zip(assembly['ucscStyleName'], assembly['refseqAccession']))
genome = pysam.FastaFile('../data/ncbi_genome/ncbi_dataset/data/GCF_000001405.13/GCF_000001405.13_GRCh37_genomic.fna')

In [ ]:
df = pd.read_csv('../data/laks2021clonal/ov2295_snv_counts.csv', usecols=['chrom', 'coord', 'ref', 'alt', 'cell_id', 'sample_id', 'alt_counts'])

/var/folders/0_/sk6g3x1d5bs17hxq132f593w0000gn/T/ipykernel_10653/2681785506.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/laks2021clonal/ov2295_snv_counts.csv', usecols=['chrom', 'coord', 'ref', 'alt', 'cell_id', 'sample_id', 'alt_counts'])


In [8]:
df = df[df['alt_counts'] > 0]
df = df.rename(columns={'chrom': 'chr', 'coord': 'start'})

In [9]:
df['sbs'] = df.progress_apply(get_sbs, axis=1, args=(genome, chr_id))

  0%|          | 0/915118 [00:00<?, ?it/s]

In [10]:
df.head()

,chr,start,ref,alt,alt_counts,cell_id,sample_id,sbs
71,1,19671432,C,T,1,SA922-A90554B-R25-C22,SA922,G[C>T]G
78,1,19671432,C,T,1,SA922-A90554B-R26-C12,SA922,G[C>T]G
106,1,19671432,C,T,1,SA922-A90554B-R30-C24,SA922,G[C>T]G
108,1,19671432,C,T,1,SA922-A90554B-R30-C64,SA922,G[C>T]G
111,1,19671432,C,T,1,SA922-A90554B-R32-C34,SA922,G[C>T]G


In [11]:
for patient_id in tqdm(df['sample_id'].unique()):
    df_patient = df[df['sample_id'] == patient_id]
    table = pd.crosstab(df_patient['cell_id'], df_patient['sbs'])
    table = table.reindex(columns=contexts['Type'], fill_value=0) # Ensure all contexts are present as columns
    table.to_csv(f'../data/laks2021clonal/laks2021clonal-{patient_id}.csv')

  0%|          | 0/3 [00:00<?, ?it/s]